In [ ]:
library(Seurat, lib.loc = "/opt/conda/lib/R/library")
library(data.table, lib.loc = "/opt/conda/lib/R/library")
library(dplyr, lib.loc = "/opt/conda/lib/R/library")

#1参数设置
raw_path <- "/data/input/Files/ResultData/Workflow"
sample1 <- "C260202001"
sample2 <- "C260202002"
out_path <- "/data/work/01AraSeed/02QC"
samples <- c(sample1,sample2)

filter_matrix <- file.path(
  raw_path,
  samples,
  samples,
  "output",
  "filter_matrix"
)
### 也可以直接写入，不建议使用此方法读取大批量数据🤖
# 创建一个向量存储所有路径
# filter_matrix <- c(
#   "/data/work/c21/output/filter_matrix",
#   "/data/work/c22/output/filter_matrix",
#   "/data/work/c23/output/filter_matrix"
# )

# 输出: /data/input/Files/ResultData/Workflow/C260202001/C260202001/output/filter_matrix
geneinfo_file <- "/data/input/Files/ReferenceData/Araport11/Arabidopsis_thaliana.geneinfo.txt"

dir.create(out_path, recursive = TRUE, showWarnings = FALSE)
setwd(out_path)


objs <- list()

for (s in filter_matrix) {
  cat(">>> Reading sample:", s, "\n")

  # 提取当前样本的样本名（在循环内，用当前的 s）
  sample_name <- basename(dirname(dirname(s)))  # 根据路径结构调整

  counts <- Read10X(data.dir = s, gene.column = 1)

  colnames(counts) <- paste(sample_name, colnames(counts), sep = ":")

  obj <- CreateSeuratObject(
    counts = counts,
    min.cells = 1,
    min.features = 1,
    project = "infertileOVU"
  )

  obj$Batch <- sample_name

  objs[[sample_name]] <- obj
}
## =========================
## 3. 合并两个文库
## =========================

scRNA <- Reduce(function(x, y) merge(x, y), objs)
rm(objs); gc()

cat("Merged object:\n")
print(table(scRNA$Batch))
## =========================
## 4. 读取基因注释，检查线粒体/叶绿体基因
## =========================

geneinfo <- fread(geneinfo_file)

# 先查看基因注释中的染色体名称
cat("基因注释中的染色体名称：\n")
print(unique(geneinfo$Chr))
# 根据实际名称调整（示例：可能是 "ChrM"/"ChrC" 或 "Mt"/"Pt"）
mt_chr <- "ChrM"  # 根据上面输出修改
cp_chr <- "ChrC"  # 根据上面输出修改

mt_genes <- geneinfo[get("Chr") == mt_chr, GeneID]
cp_genes <- geneinfo[get("Chr") == cp_chr, GeneID]

# 检查哪些基因实际存在于 scRNA 中
mt_genes <- intersect(mt_genes, rownames(scRNA))
cp_genes <- intersect(cp_genes, rownames(scRNA))

cat("线粒体基因数量：", length(mt_genes), "\n")
cat("叶绿体基因数量：", length(cp_genes), "\n")

# 计算百分比
scRNA[["percent.mt"]] <- PercentageFeatureSet(scRNA, features = mt_genes)
scRNA[["percent.C"]]  <- PercentageFeatureSet(scRNA, features = cp_genes)
## =========================
## 5. 后续 QC 步骤...
## =========================

# 从参数中获取项目名称（假设你有 project_name 参数）
# 如果没有，可以从前面的 project 参数获取
project_name <- "infertileOVU"  # 或者从变量获取

# 或者组合样本名
sample_prefix <- paste(sample_names, collapse = "_")  # sample_names 是你之前的样本列表

# 动态生成文件名
qc_file <- paste0(project_name, "_qc_stat_raw.tsv")
vln_file <- paste0(project_name, "_QC_VlnPlot_raw.pdf")
scatter_file <- paste0(project_name, "_QC_Scatter_raw.pdf")
rds_file <- paste0(project_name, "_merge_raw.rds")

# nUMIs = nCount_RNA
# nGenes = nFeature_RNA
scRNA$nUMIs  <- scRNA$nCount_RNA
scRNA$nGenes <- scRNA$nFeature_RNA

qc_stat <- rbind(
  nReads     = summary(scRNA$nCount_RNA),
  nUMIs      = summary(scRNA$nUMIs),
  nGenes     = summary(scRNA$nGenes),
  percent.mt = summary(scRNA$percent.mt),
  percent.C  = summary(scRNA$percent.C)
)

write.table(
  qc_stat,
  file = qc_file,
  sep = "\t",
  quote = FALSE,
  col.names = NA
)

## =========================
## 6. QC 小提琴图
## =========================

features <- c(
  "nCount_RNA",
  "nFeature_RNA",
  "percent.mt",
  "percent.C"
)

pdf(vln_file, width = length(features) * 4.5, height = 5)
VlnPlot(
  scRNA,
  group.by = "Batch",
  pt.size = 0,
  features = features,
  ncol = length(features)
)
dev.off()

## =========================
## 7. QC 散点图
## =========================

pdf(scatter_file, width = 7, height = 5)
for (f in c("nFeature_RNA", "percent.mt", "percent.C")) {
  p <- FeatureScatter(
    scRNA,
    group.by = "Batch",
    feature1 = "nCount_RNA",
    feature2 = f
  )
  print(p)
}
dev.off()

## =========================
## 8. 保存 Seurat 对象
## =========================

saveRDS(scRNA, file = rds_file)

cat(">>> All done. Seurat object saved as:", rds_file, "\n")
             